# 🤖 Anthropic API Usage Tracker - Google Colab Edition

**Real-time API usage monitoring with Google Drive persistence**

## 🚀 Features:
- 📱 Mobile-friendly interactive dashboards
- 💾 Automatic Google Drive data persistence
- 🔄 Real-time monitoring with configurable intervals
- 📊 Historical trend analysis
- 🚨 Balance and usage alerts
- 📥 CSV data export

## 📋 Instructions:
1. Run all cells in order
2. Authenticate with Google Drive when prompted
3. Set your Anthropic API key in the configuration section
4. Start monitoring!

---

In [ ]:
#@title 🔧 **Setup & Installation** 
#@markdown Run this cell first to install dependencies and setup the environment

# Install required packages
!pip install -q anthropic requests pandas matplotlib seaborn plotly nbformat

# Import libraries
import os
import json
import time
import requests
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import numpy as np
from datetime import datetime, timedelta
from IPython.display import display, clear_output, HTML
from google.colab import drive, files
import warnings
warnings.filterwarnings('ignore')

# Configure plotting
plt.style.use('default')  # Colab-friendly style
%matplotlib inline

print("✅ All packages installed successfully!")
print("📱 Mobile-friendly visualizations enabled")
print("🔄 Ready for Google Drive integration")

In [ ]:
#@title 🔗 **Connect to Google Drive**
#@markdown This will mount your Google Drive to persist usage data

# Mount Google Drive
drive.mount('/content/drive')

# Create project directory
project_dir = '/content/drive/MyDrive/anthropic_usage_tracker'
data_dir = f'{project_dir}/data'
os.makedirs(data_dir, exist_ok=True)

print(f"📁 Project directory: {project_dir}")
print(f"💾 Data directory: {data_dir}")
print("✅ Google Drive connected successfully!")

In [ ]:
#@title ⚙️ **Configuration**
#@markdown Set your Anthropic API key and monitoring preferences

# Configuration
API_KEY = ""  #@param {type:"string"}
REFRESH_INTERVAL_MINUTES = 5  #@param {type:"slider", min:1, max:60, step:1}
ENABLE_EMAIL_ALERTS = False  #@param {type:"boolean"}
LOW_BALANCE_THRESHOLD = 10.0  #@param {type:"number"}
HIGH_USAGE_THRESHOLD = 5.0  #@param {type:"number"}

# Save configuration
config = {
    'api_key': API_KEY,
    'refresh_interval': REFRESH_INTERVAL_MINUTES,
    'email_alerts': ENABLE_EMAIL_ALERTS,
    'low_balance_threshold': LOW_BALANCE_THRESHOLD,
    'high_usage_threshold': HIGH_USAGE_THRESHOLD,
    'last_updated': datetime.now().isoformat()
}

config_file = f'{project_dir}/config.json'
with open(config_file, 'w') as f:
    json.dump(config, f, indent=2)

print("⚙️ Configuration saved!")
if not API_KEY:
    print("⚠️  No API key set - will use demo data")
else:
    print("🔑 API key configured")

In [ ]:
class ColabAnthropicTracker:
    def __init__(self, project_dir, api_key=None):
        self.project_dir = project_dir
        self.data_dir = f'{project_dir}/data'
        self.api_key = api_key
        self.base_url = "https://api.anthropic.com/v1"
        self.history_file = f'{self.data_dir}/usage_history.json'
        self.load_history()
        
    def load_history(self):
        """Load historical data from Google Drive"""
        try:
            if os.path.exists(self.history_file):
                with open(self.history_file, 'r') as f:
                    self.history = json.load(f)
                print(f"📚 Loaded {len(self.history)} historical records")
            else:
                self.history = []
                print("📊 Starting with fresh data")
        except Exception as e:
            print(f"⚠️  Error loading history: {e}")
            self.history = []
    
    def save_history(self):
        """Save data to Google Drive"""
        try:
            with open(self.history_file, 'w') as f:
                json.dump(self.history, f, indent=2, default=str)
            print(f"💾 Saved {len(self.history)} records to Drive")
        except Exception as e:
            print(f"❌ Error saving: {e}")
    
    def fetch_usage_data(self):
        """Fetch current usage data"""
        if not self.api_key:
            print("🎭 Using demo data (no API key)")
            return self.generate_mock_data()
            
        headers = {
            'x-api-key': self.api_key,
            'anthropic-version': '2023-06-01',
            'content-type': 'application/json'
        }
        
        try:
            # Try actual API first
            response = requests.get(f"{self.base_url}/billing/usage", headers=headers, timeout=10)
            
            if response.status_code == 200:
                return response.json()
            else:
                print(f"⚠️  API returned {response.status_code}, using demo data")
                return self.generate_mock_data()
                
        except Exception as e:
            print(f"🔌 Connection issue ({e}), using demo data")
            return self.generate_mock_data()
    
    def generate_mock_data(self):
        """Generate realistic demo data"""
        now = datetime.now()
        
        # Business logic for realistic patterns
        hour = now.hour
        if 9 <= hour <= 17:  # Business hours
            base_requests = np.random.poisson(150)
            base_cost = np.random.normal(2.5, 0.8)
        elif 6 <= hour <= 9 or 17 <= hour <= 22:
            base_requests = np.random.poisson(80)
            base_cost = np.random.normal(1.2, 0.4)
        else:  # Night
            base_requests = np.random.poisson(25)
            base_cost = np.random.normal(0.3, 0.1)
        
        return {
            'timestamp': now.isoformat(),
            'requests': max(1, int(base_requests)),
            'cost_usd': max(0.01, round(base_cost, 4)),
            'tokens_input': max(100, int(np.random.normal(12000, 3000))),
            'tokens_output': max(50, int(np.random.normal(4000, 1000))),
            'balance': round(125.47 - len(self.history) * 0.1, 2)  # Decreasing balance
        }
    
    def update_and_save(self):
        """Fetch new data and save to Drive"""
        data = self.fetch_usage_data()
        if data:
            data['id'] = len(self.history) + 1
            self.history.append(data)
            self.save_history()
            return data
        return None
    
    def get_recent_df(self, hours=24):
        """Get recent data as DataFrame"""
        if not self.history:
            return pd.DataFrame()
            
        # Get recent records
        cutoff = datetime.now() - timedelta(hours=hours)
        recent = []
        
        for record in self.history[-hours:]:  # Last N records as proxy for hours
            try:
                record_time = datetime.fromisoformat(record['timestamp'].replace('Z', '+00:00'))
                if record_time >= cutoff:
                    recent.append(record)
            except:
                recent.append(record)  # Include if timestamp parsing fails
        
        if not recent:
            return pd.DataFrame()
            
        df = pd.DataFrame(recent)
        if 'timestamp' in df.columns:
            df['timestamp'] = pd.to_datetime(df['timestamp'])
            df['hour'] = df['timestamp'].dt.strftime('%H:%M')
        
        return df.tail(24)  # Last 24 records
    
    def create_mobile_dashboard(self):
        """Create mobile-friendly Plotly dashboard"""
        df = self.get_recent_df()
        
        if df.empty:
            print("📊 No data available. Run update_and_save() first.")
            return
        
        # Create subplots
        fig = make_subplots(
            rows=3, cols=2,
            subplot_titles=('💰 Hourly Costs', '📊 Request Volume', 
                          '🔤 Token Usage', '📈 Cumulative Cost',
                          '⚡ Efficiency', '💳 Balance Trend'),
            specs=[[{"secondary_y": False}, {"secondary_y": False}],
                   [{"secondary_y": False}, {"secondary_y": False}],
                   [{"secondary_y": False}, {"secondary_y": False}]]
        )
        
        # 1. Hourly Costs
        fig.add_trace(
            go.Scatter(x=df['hour'], y=df['cost_usd'], mode='lines+markers',
                      name='Cost', line=dict(color='#1f77b4', width=3)),
            row=1, col=1
        )
        
        # 2. Request Volume
        fig.add_trace(
            go.Bar(x=df['hour'], y=df['requests'], name='Requests',
                   marker_color='#ff7f0e'),
            row=1, col=2
        )
        
        # 3. Token Usage
        fig.add_trace(
            go.Bar(x=df['hour'], y=df['tokens_input'], name='Input Tokens',
                   marker_color='#2ca02c'),
            row=2, col=1
        )
        fig.add_trace(
            go.Bar(x=df['hour'], y=df['tokens_output'], name='Output Tokens',
                   marker_color='#d62728'),
            row=2, col=1
        )
        
        # 4. Cumulative Cost
        cumsum = df['cost_usd'].cumsum()
        fig.add_trace(
            go.Scatter(x=df['hour'], y=cumsum, mode='lines+markers',
                      name='Cumulative', line=dict(color='#9467bd', width=3)),
            row=2, col=2
        )
        
        # 5. Cost Efficiency
        total_tokens = df['tokens_input'] + df['tokens_output']
        efficiency = df['cost_usd'] / (total_tokens / 1000)
        fig.add_trace(
            go.Scatter(x=df['hour'], y=efficiency, mode='markers',
                      name='Cost/1K tokens', marker=dict(color='#8c564b', size=8)),
            row=3, col=1
        )
        
        # 6. Balance Trend
        fig.add_trace(
            go.Scatter(x=df['hour'], y=df['balance'], mode='lines+markers',
                      name='Balance', line=dict(color='#e377c2', width=3)),
            row=3, col=2
        )
        
        # Update layout for mobile
        fig.update_layout(
            height=1200,
            showlegend=False,
            title_text="🤖 Anthropic API Usage Dashboard",
            title_x=0.5,
            font=dict(size=10)
        )
        
        # Update x-axis labels for readability
        for i in range(1, 4):
            for j in range(1, 3):
                fig.update_xaxes(tickangle=45, row=i, col=j)
        
        fig.show()
        
        # Print summary
        self.print_summary(df)
    
    def print_summary(self, df):
        """Print usage summary with emojis"""
        if df.empty:
            return
            
        total_cost = df['cost_usd'].sum()
        total_requests = df['requests'].sum()
        current_balance = df['balance'].iloc[-1] if not df.empty else 0
        
        print("\n" + "🔥"*50)
        print("📊 REAL-TIME USAGE SUMMARY")
        print("🔥"*50)
        print(f"💰 Last 24h Cost: ${total_cost:.4f}")
        print(f"📞 Total Requests: {total_requests:,}")
        print(f"💳 Current Balance: ${current_balance:.2f}")
        
        if total_cost > 0:
            days_left = current_balance / total_cost
            print(f"⏰ Runway: {days_left:.1f} days at current rate")
            
            if current_balance < LOW_BALANCE_THRESHOLD:
                print("🚨 LOW BALANCE ALERT!")
            if total_cost > HIGH_USAGE_THRESHOLD:
                print("⚡ HIGH USAGE ALERT!")
        
        print("🔥"*50)
    
    def continuous_monitor(self, duration_hours=2):
        """Run continuous monitoring"""
        print(f"🔄 Starting {duration_hours}h monitoring session...")
        print(f"📱 Mobile-optimized for Colab")
        print("⏹️  Click 'Interrupt execution' to stop\n")
        
        end_time = time.time() + (duration_hours * 3600)
        
        try:
            while time.time() < end_time:
                clear_output(wait=True)
                
                print(f"🔄 Update: {datetime.now().strftime('%H:%M:%S')}")
                
                # Fetch and save new data
                new_data = self.update_and_save()
                if new_data:
                    print(f"📊 Balance: ${new_data['balance']:.2f} | Cost: ${new_data['cost_usd']:.4f}")
                
                # Show dashboard
                self.create_mobile_dashboard()
                
                print(f"\n⏰ Next update in {REFRESH_INTERVAL_MINUTES} minutes...")
                print("⏹️  Click 'Runtime > Interrupt execution' to stop")
                
                time.sleep(REFRESH_INTERVAL_MINUTES * 60)
                
        except KeyboardInterrupt:
            print("\n⏹️  Monitoring stopped!")
            
        print("✅ Final data saved to Google Drive")

# Initialize tracker
tracker = ColabAnthropicTracker(project_dir, API_KEY)

print("🚀 Colab Anthropic Tracker Ready!")
print("📱 Mobile-optimized for Colab environment")
print("💾 Data will persist in your Google Drive")

In [ ]:
#@title 🔄 **Quick Update & View**
#@markdown Run this to fetch latest data and show dashboard

# Quick single update
tracker.update_and_save()
tracker.create_mobile_dashboard()

In [ ]:
#@title 🎯 **Start Continuous Monitoring**
#@markdown This will run continuous monitoring for the specified duration

MONITOR_DURATION_HOURS = 2  #@param {type:"slider", min:0.5, max:12, step:0.5}

print(f"🚀 Starting {MONITOR_DURATION_HOURS} hour monitoring session...")
print("💡 Your data will be saved to Google Drive automatically")
print("📱 Dashboard optimized for mobile viewing")

tracker.continuous_monitor(duration_hours=MONITOR_DURATION_HOURS)

In [ ]:
#@title 📊 **View Historical Data**
#@markdown Analyze your historical usage patterns

# Load and display historical trends
df_all = pd.DataFrame(tracker.history)

if not df_all.empty:
    print(f"📚 Total records: {len(df_all)}")
    print(f"💰 Total spent: ${df_all['cost_usd'].sum():.2f}")
    print(f"📅 Date range: {df_all.iloc[0]['timestamp'][:10]} to {df_all.iloc[-1]['timestamp'][:10]}")
    
    # Historical trends
    fig = px.line(df_all, x='timestamp', y='cost_usd', 
                  title='📈 Historical Usage Costs',
                  labels={'cost_usd': 'Cost (USD)', 'timestamp': 'Time'})
    fig.update_layout(height=400)
    fig.show()
    
    # Balance over time
    fig2 = px.line(df_all, x='timestamp', y='balance', 
                   title='💳 Balance Over Time',
                   labels={'balance': 'Balance (USD)', 'timestamp': 'Time'})
    fig2.update_layout(height=400)
    fig2.show()
else:
    print("📊 No historical data yet. Run some monitoring first!")

In [ ]:
#@title 💾 **Export Data**
#@markdown Download your usage data as CSV

if tracker.history:
    df_export = pd.DataFrame(tracker.history)
    
    # Save to Drive
    export_file = f'{project_dir}/anthropic_usage_export.csv'
    df_export.to_csv(export_file, index=False)
    
    print(f"💾 Data exported to: {export_file}")
    print(f"📊 {len(df_export)} records exported")
    
    # Also download to local computer
    files.download(export_file)
    print("📥 File downloaded to your computer!")
else:
    print("📊 No data to export yet")